# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [ ]:
%pip install gymnasium[classic-control] stable-baselines3 wandb tsilva-notebook-utils==0.0.104 --quiet

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

In [ ]:
import torch

# TODO: move to utils
def get_default_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    elif torch.cuda.is_available(): return torch.device("cuda")
    else: return torch.device("cpu")

DEVICE = get_default_device()
print(f"Device set to: {DEVICE}")

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [ ]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        max_epochs=-1,          # Maximum number of training epochs (-1 for no limit)
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        eval_interval=5,       # Evaluate every N epochs
        eval_episodes=20,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size for networks
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
        normalize=False,       # Whether to use input normalization
        updates_per_epoch=1,      # Number of epochs to update policy per training step
        mean_reward_window=100,  # Window size for mean reward calculation (for early stopping)
        n_envs="auto"          # Maximum number of parallel environments
    )
    env_specific = {
        "CartPole-v1": dict(
            gamma=0.99,           # Standard discount for CartPole
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for CartPole
            minibatch_size=256,    # Smaller batch for faster updates
            # TODO: restore this
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=475, # Official CartPole-v1 solved threshold
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for CartPole
            entropy_coef=0.01     # Typical entropy for CartPole
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=128,       # Larger net for more complex env
            entropy_coef=0.02     # Higher entropy for more exploration
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=-100, # Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            entropy_coef=0.01     # Typical entropy for Acrobot
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200, # Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=128,       # Larger net for continuous control
            entropy_coef=0.0      # No entropy for deterministic continuous control
        ),
        "MountainCar-v0": dict(
            gamma=0.99,             # Discount factor (keep)
            lam=0.97,               # Slightly higher GAE lambda for more bias reduction
            clip_epsilon=0.15,      # Tighter PPO clip for more stable updates
            minibatch_size=16,      # Smaller minibatch for more frequent updates
            eval_interval=2,        # Keep frequent evaluation
            eval_episodes=10,       # Keep
            reward_threshold=-110,  # Keep
            policy_lr=1e-4,         # Lower learning rate for more stable policy updates
            value_lr=5e-4,          # Lower value net LR for stability
            hidden_dim=128,         # Larger network for more capacity
            entropy_coef=0.05       # Higher entropy for hard exploration
        ),
    }
    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    return {**common, **env_specific[env_id]}

ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"
CONFIG = setup_config(ENV_ID)
CONFIG

In [ ]:
# Set random seed for reproducibility
set_random_seed(CONFIG['seed'])

def _build_env(env_id, n_envs=1, seed=None, norm_obs=False, norm_reward=False):
    import multiprocessing
    from stable_baselines3.common.env_util import make_vec_env
    from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv, SubprocVecEnv
    if n_envs == 'auto': n_envs = multiprocessing.cpu_count()
    vec_env_cls = SubprocVecEnv if n_envs > 1 else DummyVecEnv
    env = make_vec_env(env_id, n_envs=n_envs, seed=seed, vec_env_cls=vec_env_cls)
    if norm_obs or norm_reward: env = VecNormalize(env, norm_obs=norm_obs, norm_reward=norm_reward)
    return env

# Wrap build env with config parameters
build_env = lambda seed, n_envs=None: _build_env(
    CONFIG['env_id'], 
    norm_obs=CONFIG['normalize'], 
    n_envs=n_envs if n_envs is not None else CONFIG['n_envs'], seed=seed
)

# Test building env
env = build_env(CONFIG['seed'])
# TODO: encapsulate this
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, 1)
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
from __future__ import annotations

from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.distributions import Categorical
import numpy as np
import torch
from torch.distributions import Categorical
from typing import Optional, Tuple, Union, Sequence

from typing import Optional, Sequence, Tuple
import numpy as np
import torch
from torch.distributions import Categorical

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def _device_of(module: torch.nn.Module) -> torch.device:  # type: ignore
    """Return the device of *module*'s first parameter."""
    return next(module.parameters()).device


# -----------------------------------------------------------------------------
# Rollout collector (env‑major)
# -----------------------------------------------------------------------------

def collect_rollouts(
    env,
    policy_model: torch.nn.Module,
    value_model: Optional[torch.nn.Module] = None,
    n_steps: int = 1024,
    deterministic: bool = False,
    gamma: float = 0.99,
    lam: float = 0.95,
    normalize_advantage: bool = True,
    adv_norm_eps: float = 1e-8,
    collect_frames: bool = False,
):
    """Collect *n_steps* transitions from *env* in parallel and return env‑major tensors."""

    device: torch.device = _device_of(policy_model)
    n_envs: int = env.num_envs

    # ------------------------------------------------------------------
    # 1. Buffers (time, env) for easy appending during rollout
    # ------------------------------------------------------------------
    obs_buf: list[np.ndarray] = []       # build list first, stack later
    act_buf = np.zeros((n_steps, n_envs), np.int64)
    rew_buf = np.zeros((n_steps, n_envs), np.float32)
    done_buf = np.zeros((n_steps, n_envs), bool)
    logp_buf = np.zeros((n_steps, n_envs), np.float32)
    val_buf = np.zeros((n_steps, n_envs), np.float32)
    frame_buf: list[Sequence[np.ndarray]] | None = [] if collect_frames else None

    # ------------------------------------------------------------------
    # 2. Rollout
    # ------------------------------------------------------------------
    obs = env.reset()
    for t in range(n_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad(): logits = policy_model(obs_t)
        dist = Categorical(logits=logits)

        act_t = logits.argmax(-1) if deterministic else dist.sample()
        logp_t = dist.log_prob(act_t)
        val_t = (
            value_model(obs_t).squeeze(-1) if value_model is not None else torch.zeros(n_envs, device=device)
        )

        next_obs, reward, done, infos = env.step(act_t.cpu().numpy())

        # store
        obs_buf.append(obs.copy())
        act_buf[t], rew_buf[t], done_buf[t] = act_t.cpu().numpy(), reward, done
        logp_buf[t] = logp_t.detach().cpu().numpy()
        val_buf[t] = val_t.detach().cpu().numpy()

        if collect_frames and frame_buf is not None:
            frame_buf.append(env.get_images())

        obs = next_obs

    # ------------------------------------------------------------------
    # 3. Bootstrap value for the next state of each env
    # ------------------------------------------------------------------
    with torch.no_grad():
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
        next_values = (
            value_model(obs_t).squeeze(-1).cpu().numpy()
            if value_model is not None else np.zeros(n_envs, dtype=np.float32)
        )

    # ------------------------------------------------------------------
    # 4. GAE‑λ advantage / return (with correct masking)
    # ------------------------------------------------------------------
    adv_buf = np.zeros_like(rew_buf)
    ret_buf = np.zeros_like(rew_buf)

    gae = np.zeros(n_envs, dtype=np.float32)
    next_non_terminal = 1.0 - done_buf[-1].astype(np.float32)
    next_value = next_values

    for t in reversed(range(n_steps)):
        delta = rew_buf[t] + gamma * next_value * next_non_terminal - val_buf[t]
        gae = delta + gamma * lam * next_non_terminal * gae
        adv_buf[t] = gae

        next_non_terminal = 1.0 - done_buf[t].astype(np.float32)
        next_value = val_buf[t]

    ret_buf = adv_buf + val_buf

    if normalize_advantage:
        adv_flat = adv_buf.reshape(-1)
        adv_buf = (adv_buf - adv_flat.mean()) / (adv_flat.std() + adv_norm_eps)

    # ------------------------------------------------------------------
    # 5. Env‑major flattening: (time, env, …) -> (env, time, …) -> (env*time, …)
    # ------------------------------------------------------------------
    obs_arr = np.stack(obs_buf)                       # (T, E, obs)
    obs_env_major = obs_arr.transpose(1, 0, 2)        # (E, T, obs)
    states = torch.as_tensor(obs_env_major.reshape(n_envs * n_steps, -1), dtype=torch.float32)

    def _flat_env_major(arr: np.ndarray, dtype: torch.dtype):
        return torch.as_tensor(arr.transpose(1, 0).reshape(-1), dtype=dtype)

    actions = _flat_env_major(act_buf, torch.int64)
    rewards = _flat_env_major(rew_buf, torch.float32)
    dones = _flat_env_major(done_buf, torch.bool)
    logps = _flat_env_major(logp_buf, torch.float32)
    values = _flat_env_major(val_buf, torch.float32)
    advs = _flat_env_major(adv_buf, torch.float32)
    returns = _flat_env_major(ret_buf, torch.float32)

    if collect_frames and frame_buf is not None:
        # frame_buf is list[T] where each item is list[E] of images
        frames_env_major: list[np.ndarray] = []
        for e in range(n_envs):
            for t in range(n_steps):
                frames_env_major.append(frame_buf[t][e])
    else:
        frames_env_major = [0] * (n_envs * n_steps)

    # ------------------------------------------------------------------
    # 6. Return in the SB3‑compatible order
    # ------------------------------------------------------------------
    return (
        states,
        actions,
        rewards,
        dones,
        logps,
        values,
        advs,
        returns,
        frames_env_major,
    )

policy_model = PolicyNet(obs_dim, act_dim, hidden_dim=CONFIG['hidden_dim']).to(DEVICE)
value_model = ValueNet(obs_dim, hidden_dim=CONFIG['hidden_dim']).to(DEVICE)
trajectories = collect_rollouts( # trajectories = (states, actions, rewards, dones, logps, values, adv, returns)
    env,
    policy_model,
    value_model,
    deterministic=False,
    collect_frames=False
)
len(trajectories)
trajectories[0].shape, trajectories[1].shape, trajectories[2].shape, trajectories[3].shape, \
trajectories[4].shape, trajectories[5].shape, trajectories[6].shape, trajectories[7].shape, \
len(trajectories[8])  # frames
# trajectories[0] = states, actions, rewards, dones, logps, values,

In [ ]:
def group_trajectories_by_episode(trajectories):
    episodes = []
    episode = []

    T = trajectories[0].shape[0]  # number of time steps

    for t in range(T):
        step = tuple(x[t] for x in trajectories)  # (state, action, reward, done, ...)
        episode.append(step)
        done = step[3]
        if done.item():  # convert tensor to bool
            episodes.append(episode)
            episode = []

    return episodes

episodes = group_trajectories_by_episode(trajectories)

# Convert rewards to float
total_episode_rewards = [sum(ep[2].item() for ep in episode) for episode in episodes]

# Optional: print for inspection
for i, r in enumerate(total_episode_rewards):
    print(f"Episode {i}: Total Reward = {r}")

In [ ]:
from torch.utils.data import Dataset

# TODO: should dataloader move to gpu?
class RolloutDataset(Dataset):
    """Holds PPO roll-out tensors and lets them be swapped in-place."""
    def __init__(self):
        self.trajectories = None 

    def update(self, *trajectories):
        self.trajectories = trajectories

    def __len__(self):
        return len(self.trajectories[0])

    def __getitem__(self, idx):
        return tuple(t[idx] for t in self.trajectories)


In [ ]:
import torch
import multiprocessing
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from torch.distributions import Categorical
from collections import deque

# ---------------------------------------------------------------------
#  PPO Lightning module
#  (assumes PolicyNet, ValueNet, build_env, collect_rollouts are defined)
# ---------------------------------------------------------------------
class PPOAgent(pl.LightningModule):
    def __init__(self, obs_dim, act_dim, config):
        super().__init__()
        self.save_hyperparameters()

        # ----------------- unpack config -----------------
        self.config = config
        self.entropy_coef   = config['entropy_coef']
        self.clip_epsilon   = config['clip_epsilon']
        self.gamma          = config['gamma']
        self.lam            = config['lam']
        self.minibatch_size     = config['minibatch_size']
        self.eval_interval      = config['eval_interval']
        self.eval_episodes      = config['eval_episodes']
        self.reward_threshold   = config['reward_threshold']
        self.updates_per_epoch  = config['updates_per_epoch']
        self.policy_lr          = config['policy_lr']
        self.value_lr           = config['value_lr']
        self.mean_reward_window = config['mean_reward_window']

        # ----------------- models & env -----------------
        self.policy_model = PolicyNet(
            obs_dim, act_dim,
            hidden_dim=config['hidden_dim']# TODO: read from params
        )
        self.value_model = ValueNet(
            obs_dim,
            hidden_dim=config['hidden_dim']# TODO: read from params
        )
        self.env = build_env(config['seed']) # TODO: read from params
        self.obs_dim, self.act_dim = obs_dim, act_dim

        # ----------------- rollout storage --------------
        self.rollout_ds = RolloutDataset()   # <-- created once

        # TODO: softcode
        self.episode_reward_deque = deque(maxlen=self.mean_reward_window)  # Store recent mean rewards for early stopping
        
        # we’ll step the two optimizers manually
        self.automatic_optimization = False

    # ===================================================
    #  Lightning hooks
    # ===================================================
    def setup(self, stage: str):
        """Collect an initial roll-out before dataloaders are requested."""
        if stage == "fit": self._collect_and_store_rollout()   # fills rollout_ds

    # TODO: is this called only once?
    def train_dataloader(self):
        """Standard DataLoader built *once*; dataset is mutable."""
        return DataLoader(
            self.rollout_ds,
            batch_size=self.minibatch_size,
            shuffle=True,
            pin_memory=True if self.device.type != 'mps' else False,  # Set to False for MPS compatibility
            # TODO: doesn't converge when True (memory sharing issues?)
            #persistent_workers=False,
            num_workers=multiprocessing.cpu_count() // 2 if self.device.type != 'mps' else 0
        )

    def on_train_epoch_start(self):
        """Refresh roll-out tensors in-place each epoch."""
        self._collect_and_store_rollout()

    # ---------------------------------------------------
    def training_step(self, batch, batch_idx):
        opt_policy, opt_value = self.optimizers()

        # unpack batch
        (states, actions, rewards, dones, old_logps, values, advantages, returns_, frames) = batch

        # main PPO loop
        policy_losses, value_losses = [], []
        clip_fractions, entropy_vals, kl_divs, approx_kl_divs = [], [], [], []
        
        for _ in range(self.updates_per_epoch):
            logits = self.policy_model(states)
            dist   = Categorical(logits=logits)
            new_logps = dist.log_prob(actions)

            ratio = torch.exp(new_logps - old_logps)
            surr1 = ratio * advantages
            surr2 = torch.clamp(
                ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon
            ) * advantages
            entropy = dist.entropy().mean()

            # === PPO DEBUG METRICS ===
            # 1. Clipping fraction - how often is clipping happening?
            clip_fraction = ((ratio < 1.0 - self.clip_epsilon) | (ratio > 1.0 + self.clip_epsilon)).float().mean()
            
            # 2. KL divergence between old and new policies
            kl_div = (old_logps - new_logps).mean()  # Simple KL approximation
            approx_kl = ((ratio - 1) - torch.log(ratio)).mean()  # Better KL approximation
            
            # 3. Explained variance of value function
            value_pred = self.value_model(states).squeeze()
            explained_var = 1 - torch.var(returns_ - value_pred) / torch.var(returns_)

            policy_loss = -torch.min(surr1, surr2).mean()
            policy_loss -= self.entropy_coef * entropy
            value_loss = 0.5 * ((returns_ - value_pred) ** 2).mean()

            # Store metrics
            clip_fractions.append(clip_fraction.detach())
            entropy_vals.append(entropy.detach())
            kl_divs.append(kl_div.detach())
            approx_kl_divs.append(approx_kl.detach())

            # ---------- optimizers ----------
            opt_policy.zero_grad()
            self.manual_backward(policy_loss)
            opt_policy.step()

            opt_value.zero_grad()
            self.manual_backward(value_loss)
            opt_value.step()

            policy_losses.append(policy_loss.detach())
            value_losses.append(value_loss.detach())

        # Calculate means
        mean_policy_loss = torch.stack(policy_losses).mean()
        mean_value_loss = torch.stack(value_losses).mean()
        mean_clip_fraction = torch.stack(clip_fractions).mean()
        mean_entropy = torch.stack(entropy_vals).mean()
        mean_kl = torch.stack(kl_divs).mean()
        mean_approx_kl = torch.stack(approx_kl_divs).mean()

        # Additional useful metrics
        advantage_mean = advantages.mean()
        advantage_std = advantages.std()
        value_mean = values.mean()
        returns_mean = returns_.mean()

        mean_reward = np.mean(self.episode_reward_deque) if len(self.episode_reward_deque) >= self.mean_reward_window else None

        # Enhanced logging
        log_dict = {
            'train/policy_loss': mean_policy_loss,
            'train/value_loss': mean_value_loss,
            'train/entropy': mean_entropy,
            'train/kl_divergence': mean_kl,
            'train/explained_variance': explained_var
        }
        if mean_reward is not None:  log_dict['train/mean_reward'] = mean_reward
        self.log_dict(log_dict, prog_bar=True)
        
        log_dict = {
            'train/approx_kl': mean_approx_kl,
            'train/clip_fraction': mean_clip_fraction,
            'train/advantage_mean': advantage_mean,
            'train/advantage_std': advantage_std,
            'train/value_mean': value_mean,
            'train/returns_mean': returns_mean,
        }
        self.log_dict(log_dict, prog_bar=False)

        # Early stopping with KL divergence safety
        if mean_approx_kl > 0.02:  # Common threshold
            print(f"Early stopping due to high KL divergence: {mean_approx_kl:.4f}")
            # Optionally break the inner loop early
        
        if mean_reward is not None and mean_reward >= self.reward_threshold:
            print(f"Early stopping at epoch {self.current_epoch} with mean reward {mean_reward:.2f} >= threshold {self.reward_threshold}")
            self.trainer.should_stop = True

        return mean_policy_loss + mean_value_loss

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.policy_model.parameters(), lr=self.policy_lr),
            torch.optim.Adam(self.value_model.parameters(), lr=self.value_lr)
        ]

    # ===================================================
    #  helpers
    # ===================================================
    def _collect_and_store_rollout(self):
        """Run episodes and refresh stored tensors on the model’s device."""
        
        # Collect rollouts and add them to 
        # # dataset for sampling during training step
        trajectories = collect_rollouts(
            self.env,
            self.policy_model,
            self.value_model
        )
        self.rollout_ds.update(*trajectories)

        # Add mean episode rewards to deque 
        # (to be able to average over N episodes)
        episodes = group_trajectories_by_episode(trajectories)
        episode_rewards = [sum(step[2] for step in episode) for episode in episodes]
        for r in episode_rewards: self.episode_reward_deque.append(float(r))
    
    # ---------------------------------------------------
    def forward(self, x):
        return self.policy_model(x)


## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

# Create PPO agent and move to device
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project="gymnasium_ppo") # TODO: softcode this

trainer = Trainer(
    logger=wandb_logger,
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=1,
    enable_progress_bar=True,
    accelerator="auto"
)

# Fit the model
trainer.fit(ppo_agent)

In [ ]:
import random
trajectories = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=4),
    ppo_agent.policy_model,
    n_steps=500,
    deterministic=True,
    collect_frames=True
)

In [ ]:
from tsilva_notebook_utils.gymnasium import render_episode_frames
import random

# TODO: encapsulate this
# TODO: bug received episodes is bigger than requested n_episodes
trajectories = collect_rollouts(
    build_env(random.randint(0, 1_000_000), n_envs=4),
    ppo_agent.policy_model,
    n_steps=500,
    deterministic=True,
    collect_frames=True
)
print(len(trajectories)) # should be 9
episodes = group_trajectories_by_episode(trajectories) # something is wrong in frame collection
print(len(episodes))
mean_reward = np.mean([sum(step[2] for step in episode) for episode in episodes])
episode_frames = [[step[-1] for step in episode] for episode in episodes]
print(len(episode_frames[0]))
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(episode_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))

TODO: stop on eval reward instead
TODO: fix episode grouping issue that is causing >500 frames in episode 1
TODO: figure out correct number of updates per epoch for PPO